# PIM-PAM Document Translation Pipeline — WBG Translation API

## Overview

This notebook translates PDF (and other Office) documents using the
**World Bank Group Document Translation API** (`POST /api/Translate/File`).
Unlike the Mistral pipeline, which extracts raw text and sends it to an LLM,
this API accepts the **original binary file** and returns a fully translated
document — preserving layout, tables, headers, and formatting.

### How the API works

| Input | Output |
|---|---|
| `.pdf` | `.pdf` (translated) |
| `.doc`, `.odt`, `.rtf` | `.docx` |
| `.xls`, `.ods` | `.xlsx` |
| `.ppt`, `.odp` | `.pptx` |

### Workflow

1. **Credential setup** — Fetches an OAuth 2.0 bearer token from Azure AD
   using a service principal stored in Databricks Secrets.
2. **Single-file translation** — Uploads a file via `multipart/form-data` to
   the WBG Translation API and saves the translated binary response.
3. **Bulk translation** — Iterates over all supported files in a directory,
   skipping already-translated outputs for resumability.
4. **Audit log** — Prints a structured summary of successes, skips, and failures.

### Key API parameters

| Parameter | Type | Description |
|---|---|---|
| `ToLanguage` | string | Target language code (e.g. `en`, `fr`, `es`) |
| `FromLanguage` | string | Source language code (e.g. `ko`, `ar`, `fr`) |
| `File` | binary | The document file to translate |
| `SourceStorageURI` | string | *(Optional)* Azure Blob source container URL |
| `TargetStorageURI` | string | *(Optional)* Azure Blob target container URL |
| `CategoryOverride` | string | *(Optional)* Custom translator category (e.g. ICSID model) |

### Authentication

Azure AD service principal via `azure-identity`.
Secrets are stored in Databricks Secret scope `DAPGPTKEYVAULT`.

| Secret key | Description |
|---|---|
| `WBG-Translate-Tenant-ID` | Azure AD tenant ID |
| `WBG-Translate-Client-ID` | Service principal client ID |
| `WBG-Translate-Client-Secret` | Service principal client secret |

> **Note:** The Client ID and Scope for the Dev environment are hardcoded
> below as constants since they are non-sensitive identifiers, not secrets.
 

In [0]:
%pip install pypdf python-slugify azure-identity requests

In [0]:
import os
import re
import logging
import time
import mimetypes
from pathlib import Path
from datetime import datetime
 
import requests
from slugify import slugify
from azure.identity import ClientSecretCredential
 
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

## Configuration

Edit the variables in this cell to point to your input/output directories
and choose the source/target language pair.

### Language codes
Use BCP-47 / ISO 639-1 codes as expected by Azure Cognitive / Translator service:
`en`, `fr`, `es`, `ar`, `ko`, `zh-Hans`, `zh-Hant`, `pt`, `ru`, `de`, `ja`, …

### Supported input extensions
The API handles: `.pdf`, `.docx`, `.doc`, `.odt`, `.rtf`,
                 `.xlsx`, `.xls`, `.ods`, `.pptx`, `.ppt`, `.odp`

In [0]:
INPUT_DIR = "/Volumes/prd_mega/saiana95/vaiana95/Translation/input_documents/"
 
# Directory where translated files will be written
OUTPUT_DIR = "/Volumes/prd_mega/saiana95/vaiana95/Translation/translation_wbg/"

# ── Language pair ─────────────────────────────────────────────────────────────
FROM_LANGUAGE = "ko"   # Source language code  (e.g. "ko" for Korean)
TO_LANGUAGE   = "en"   # Target language code  (e.g. "en" for English)
 
# ── Optional API parameters ───────────────────────────────────────────────────
# Leave as None to use the API defaults
CATEGORY_OVERRIDE     = None   # e.g. "ICSID" — custom translator model category
SOURCE_STORAGE_URI    = None   # Azure Blob URL for source  (batch/storage-linked mode)
TARGET_STORAGE_URI    = None   # Azure Blob URL for target  (batch/storage-linked mode)
 
# ── Retry configuration ───────────────────────────────────────────────────────
MAX_RETRIES           = 3
RETRY_BACKOFF_SECONDS = 5
REQUEST_TIMEOUT_SEC   = 300    # File translation can be slow; 5 min timeout
 
# ── Supported input extensions ────────────────────────────────────────────────
SUPPORTED_EXTENSIONS = {
    ".pdf", ".docx", ".doc", ".odt", ".rtf",
    ".xlsx", ".xls", ".ods",
    ".pptx", ".ppt", ".odp",
}
 
# ── Extension → translated output extension mapping ───────────────────────────
# Matches the API's documented conversion behaviour
OUTPUT_EXTENSION_MAP = {
    ".pdf":  ".pdf",
    ".docx": ".docx", ".doc": ".docx", ".odt": ".docx", ".rtf": ".docx",
    ".xlsx": ".xlsx", ".xls": ".xlsx", ".ods": ".xlsx",
    ".pptx": ".pptx", ".ppt": ".pptx", ".odp": ".pptx",
}

## Azure AD Authentication

A `ClientSecretCredential` is constructed from secrets stored in the
Databricks Secret scope `DAPGPTKEYVAULT`.  A fresh bearer token is
fetched on every API call so long-running batch jobs never expire mid-way.

**Client ID** and **Scope** for the Dev environment are fixed constants
(public identifiers, not secrets)

In [0]:
WBG_TRANSLATE_CLIENT_ID = "<WBG_TRANSLATE_CLIENT_ID>"
WBG_TRANSLATE_SCOPE = (
    "<WBG_TRANSLATE_SCOPE>"
)
 
# WBG Translation API base URL
WBG_TRANSLATE_URL = "https://wbgtranslatedev.worldbank.org/api/Translate/File"
 
# ── Build credential from Databricks Secrets ──────────────────────────────────
credential = ClientSecretCredential(
    tenant_id     = dbutils.secrets.get(scope="<>", key="WBG-Translate-Tenant-ID"),
    client_id     = "YOUR_CLIENT_ID",
    client_secret = dbutils.secrets.get(scope="<>", key="WBG-Translate-Client-Secret"),
)
 
logger.info("Azure AD credential object created successfully.")
print("Azure AD credential object created successfully")

## Helper Functions

| Function | Description |
|---|---|
| `get_bearer_token()` | Fetches a fresh OAuth 2.0 token from Azure AD |
| `get_content_type(file_path)` | Returns the MIME type for a given file extension |
| `build_output_path(file_path, output_dir)` | Constructs a safe output path with the correct translated extension |
| `list_translatable_files(input_dir)` | Lists all supported files in a directory, sorted |
 

In [0]:
def get_bearer_token() -> str:
    """
    Fetch a fresh Azure AD OAuth 2.0 bearer token.
 
    Uses the module-level ``credential`` (ClientSecretCredential).
    Tokens are valid for ~1 hour; fetching per call ensures long batch
    jobs never expire mid-document.
 
    Returns:
        Bearer token string.
    """
    return credential.get_token(WBG_TRANSLATE_SCOPE).token
 
 
def get_content_type(file_path: str) -> str:
    """
    Return the MIME type for a file based on its extension.
 
    Falls back to ``application/octet-stream`` for unknown types.
 
    Args:
        file_path: Path to the file.
 
    Returns:
        MIME type string (e.g. ``"application/pdf"``).
    """
    ext = Path(file_path).suffix.lower()
    mime_map = {
        ".pdf":  "application/pdf",
        ".docx": "application/vnd.openxmlformats-officedocument.wordprocessingml.document",
        ".doc":  "application/msword",
        ".odt":  "application/vnd.oasis.opendocument.text",
        ".rtf":  "application/rtf",
        ".xlsx": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        ".xls":  "application/vnd.ms-excel",
        ".ods":  "application/vnd.oasis.opendocument.spreadsheet",
        ".pptx": "application/vnd.openxmlformats-officedocument.presentationml.presentation",
        ".ppt":  "application/vnd.ms-powerpoint",
        ".odp":  "application/vnd.oasis.opendocument.presentation",
    }
    return mime_map.get(ext, "application/octet-stream")
 
 
def build_output_path(file_path: str, output_dir: str, suffix_tag: str = "_WBG_ENG") -> str:
    """
    Build a filesystem-safe output path for the translated document.
 
    The stem is passed through ``slugify`` (matching the Mistral pipeline
    convention) and the extension is mapped to the API's output format.
    A ``suffix_tag`` is appended before the extension to distinguish
    translated outputs from originals.
 
    Example
    -------
    ``"ORI_Enforcement Decree_Korean.pdf"``
    → ``"ori-enforcement-decree-korean_WBG_ENG.pdf"``
 
    Args:
        file_path:   Path to the source file.
        output_dir:  Directory where the output will be saved.
        suffix_tag:  Tag appended after the slug (default ``"_WBG_ENG"``).
 
    Returns:
        Absolute output file path string.
    """
    src = Path(file_path)
    ext_in  = src.suffix.lower()
    ext_out = OUTPUT_EXTENSION_MAP.get(ext_in, ext_in)   # e.g. .doc → .docx
    slug    = slugify(src.stem)                            # e.g. "ori-enforcement-decree-korean"
    out_name = f"{slug}{suffix_tag}{ext_out}"              # e.g. "ori-..._WBG_ENG.pdf"
    return os.path.join(output_dir, out_name)
 
 
def list_translatable_files(input_dir: str) -> list:
    """
    Return a sorted list of absolute paths for all supported files in a directory.
 
    Only extensions in ``SUPPORTED_EXTENSIONS`` are included.
 
    Args:
        input_dir: Directory to scan.
 
    Returns:
        Sorted list of absolute file path strings.
    """
    files = [
        os.path.join(input_dir, f)
        for f in os.listdir(input_dir)
        if Path(f).suffix.lower() in SUPPORTED_EXTENSIONS
    ]
    return sorted(files)

## Core Translation Function — `translate_file_wbg()`

This is the heart of the pipeline. It:

1. Opens the source file as binary.
2. Builds a `multipart/form-data` request with `File`, `ToLanguage`,
   `FromLanguage`, and any optional parameters set in Cell 4.
3. Attaches a fresh Azure AD bearer token on every call.
4. POSTs to `POST /api/Translate/File`.
5. Reads the translated binary from the response body.
6. Infers the output filename from the `Content-Disposition` response header
   (falling back to `build_output_path()` if the header is absent).
7. Writes the binary to `output_dir` and returns the saved path.

### Retry logic

Transient HTTP 5xx errors and network timeouts are retried up to `MAX_RETRIES`
times with linear back-off (`RETRY_BACKOFF_SECONDS × attempt`).
 

In [0]:
def translate_file_wbg(
    file_path: str,
    output_dir: str,
    from_language: str = FROM_LANGUAGE,
    to_language: str   = TO_LANGUAGE,
    output_path: str | None = None,
) -> str:
    """
    Translate a document using the WBG Document Translation API.
 
    Uploads the file as ``multipart/form-data`` to
    ``POST /api/Translate/File`` and saves the translated binary
    response to ``output_dir``.
 
    Args:
        file_path:     Absolute path to the source document.
        output_dir:    Directory where the translated file is saved.
        from_language: BCP-47 source language code (default: ``FROM_LANGUAGE``).
        to_language:   BCP-47 target language code (default: ``TO_LANGUAGE``).
        output_path:   Optional explicit output path.  If ``None``, the path is
                       derived via ``build_output_path()``.
 
    Returns:
        Absolute path to the saved translated file.
 
    Raises:
        FileNotFoundError: If ``file_path`` does not exist.
        RuntimeError:      If all retry attempts fail.
    """
    src = Path(file_path)
    if not src.exists():
        raise FileNotFoundError(f"Source file not found: {file_path}")
 
    os.makedirs(output_dir, exist_ok=True)
    content_type = get_content_type(file_path)
    dest_path    = output_path or build_output_path(file_path, output_dir)
 
    logger.info(f"  Translating : {src.name}  ({from_language} → {to_language})")
    logger.info(f"  Content-Type: {content_type}")
 
    last_exc = None
 
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            # ── 1. Fresh bearer token ─────────────────────────────────────
            token = get_bearer_token()
            headers = {"Authorization": f"Bearer {token}"}
 
            # ── 2. Build multipart form data ──────────────────────────────
            with open(file_path, "rb") as fh:
                file_bytes = fh.read()
 
            # Form fields — only include optional params when provided
            form_data = {
                "ToLanguage":   (None, to_language),
                "FromLanguage": (None, from_language),
                "File":         (src.name, file_bytes, content_type),
            }
            if SOURCE_STORAGE_URI:
                form_data["SourceStorageURI"] = (None, SOURCE_STORAGE_URI)
            if TARGET_STORAGE_URI:
                form_data["TargetStorageURI"] = (None, TARGET_STORAGE_URI)
            if CATEGORY_OVERRIDE:
                form_data["CategoryOverride"] = (None, CATEGORY_OVERRIDE)
 
            # ── 3. POST request ───────────────────────────────────────────
            response = requests.post(
                WBG_TRANSLATE_URL,
                headers=headers,
                files=form_data,
                timeout=REQUEST_TIMEOUT_SEC,
            )
            response.raise_for_status()
 
            # ── 4. Extract filename from Content-Disposition (if present) ─
            # e.g. Content-Disposition: attachment; filename="report_en.pdf"
            cd_header = response.headers.get("Content-Disposition", "")
            cd_match  = re.search(r'filename="?([^";]+)"?', cd_header)
            if cd_match:
                api_filename = cd_match.group(1).strip()
                dest_path = os.path.join(output_dir, api_filename)
                logger.info(f"  API filename: {api_filename}")
 
            # ── 5. Write translated binary to disk ────────────────────────
            with open(dest_path, "wb") as out_fh:
                out_fh.write(response.content)
 
            logger.info(f"  Saved       : {dest_path}  ({len(response.content):,} bytes)")
            return dest_path
 
        except requests.exceptions.HTTPError as exc:
            status = exc.response.status_code if exc.response is not None else "?"
            logger.warning(f"  HTTP {status} on attempt {attempt}/{MAX_RETRIES}: {exc}")
            last_exc = exc
        except requests.exceptions.RequestException as exc:
            logger.warning(f"  Network error on attempt {attempt}/{MAX_RETRIES}: {exc}")
            last_exc = exc
 
        if attempt < MAX_RETRIES:
            wait = RETRY_BACKOFF_SECONDS * attempt
            logger.info(f"  Retrying in {wait}s …")
            time.sleep(wait)
 
    raise RuntimeError(
        f"All {MAX_RETRIES} retry attempts failed for '{src.name}'. "
        f"Last error: {last_exc}"
    )

## Single-File Translation — Demo

Use this cell to test the pipeline on **one document** before running
the bulk batch.  Set `SINGLE_FILE_PATH` to the absolute path of the file
you want to translate.

The translated file will be saved to `OUTPUT_DIR` with the naming pattern:
`<slugified-stem>_WBG_ENG.<ext>`

In [0]:
SINGLE_FILE_PATH = "/Volumes/prd_mega/saiana95/vaiana95/Translation/korean_documents/ORI_National Finance Law_Korean.pdf"
 
logger.info("=" * 60)
logger.info(f"Single-file demo: {Path(SINGLE_FILE_PATH).name}")
logger.info(f"Language pair   : {FROM_LANGUAGE} → {TO_LANGUAGE}")
logger.info("=" * 60)
 
try:
    result = translate_file_wbg(
        file_path     = SINGLE_FILE_PATH,
        output_dir    = OUTPUT_DIR,
        from_language = FROM_LANGUAGE,
        to_language   = TO_LANGUAGE,
    )
    print(f"\n✅ Translation complete.\nOutput saved to: {result}")
 
except FileNotFoundError as e:
    print(f"\n❌ File not found: {e}")
except RuntimeError as e:
    print(f"\n❌ Translation failed: {e}")

## Bulk Translation Runner

This cell iterates over every supported file in `INPUT_DIR` and calls
`translate_file_wbg()` for each one.

### Skip logic

If the expected output path already exists in `OUTPUT_DIR`, that file is
skipped without calling the API.  This makes the batch **resumable** —
re-running after a partial failure picks up from where it left off.

### Audit log

Results are collected in `audit_log` (a list of dicts) and displayed as a
summary table in Cell 10.

### Configuration recap

| Variable | Description |
|---|---|
| `INPUT_DIR` | Source document directory (set in Cell 4) |
| `OUTPUT_DIR` | Translated file output directory (set in Cell 4) |
| `FROM_LANGUAGE` | Source language code (set in Cell 4) |
| `TO_LANGUAGE` | Target language code (set in Cell 4) |
| `MAX_RETRIES` | API retry attempts per file (set in Cell 4) |

In [0]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
 
# ── Discover input files ───────────────────────────────────────────────────────
input_files = list_translatable_files(INPUT_DIR)
 
if not input_files:
    logger.warning(f"No supported files found in: {INPUT_DIR}")
else:
    logger.info(f"Found {len(input_files)} file(s) to process.")
 
# ── Audit log ─────────────────────────────────────────────────────────────────
audit_log: list[dict] = []
 
for file_path in input_files:
    filename    = Path(file_path).name
    dest_path   = build_output_path(file_path, OUTPUT_DIR)
 
    logger.info("=" * 60)
    logger.info(f"Processing : {filename}")
    logger.info(f"Started at : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
 
    # ── Skip if output already exists ─────────────────────────────────────────
    if os.path.exists(dest_path):
        logger.info(f"Skipping   : output already exists → {dest_path}")
        audit_log.append({
            "file":        filename,
            "status":      "skipped",
            "output":      dest_path,
            "size_bytes":  os.path.getsize(dest_path),
            "error":       None,
            "finished_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        })
        continue
 
    # ── Translate ──────────────────────────────────────────────────────────────
    try:
        result_path = translate_file_wbg(
            file_path     = file_path,
            output_dir    = OUTPUT_DIR,
            from_language = FROM_LANGUAGE,
            to_language   = TO_LANGUAGE,
        )
        finished_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        logger.info(f"Done       : {filename} → {result_path}")
        logger.info(f"Finished at: {finished_at}")
        audit_log.append({
            "file":        filename,
            "status":      "translated",
            "output":      result_path,
            "size_bytes":  os.path.getsize(result_path),
            "error":       None,
            "finished_at": finished_at,
        })
 
    except Exception as exc:
        logger.error(f"FAILED     : {filename} — {exc}")
        audit_log.append({
            "file":        filename,
            "status":      "failed",
            "output":      None,
            "size_bytes":  None,
            "error":       str(exc),
            "finished_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        })
 
logger.info("=" * 60)
logger.info("Batch complete.")

## Audit Log

Summary of the batch run.

| Column | Description |
|---|---|
| `file` | Source filename |
| `status` | `translated`, `skipped`, or `failed` |
| `output` | Path to translated file (if produced) |
| `size_bytes` | Size of output file in bytes |
| `error` | Error message if `status == "failed"` |
| `finished_at` | Timestamp when processing completed |

In [0]:
import pandas as pd
 
audit_df = pd.DataFrame(audit_log)
 
# ── Summary counts ─────────────────────────────────────────────────────────────
if not audit_df.empty:
    counts = audit_df["status"].value_counts().to_dict()
    total  = len(audit_df)
    print(f"\n{'='*60}")
    print(f"  Batch Summary ({total} file(s))")
    print(f"{'='*60}")
    print(f"  ✅ Translated : {counts.get('translated', 0)}")
    print(f"  ⏭  Skipped    : {counts.get('skipped', 0)}")
    print(f"  ❌ Failed     : {counts.get('failed', 0)}")
    print(f"{'='*60}\n")
 
    # Full detail table
    display(
        audit_df[[
            "file", "status", "size_bytes", "output", "error", "finished_at"
        ]]
    )
else:
    print("No files were processed.")